# Evaluator-Optimizer Loop: Python Code Assessment Subgraph

This notebook demonstrates and verifies the **Evaluator-Optimizer** workflow for the **Assessor Agent** (`US-05`, `US-06`, and `US-07`) in the **Sprints Multi-Agent AI Career & Assessment Orchestrator**.

### Architecture & Execution Sequence
1. **Code Extraction Node (`extract_code`):** Parses clean Python code from Markdown fences or raw text.
2. **Deterministic Python REPL Execution (`run_repl`):** Executes student code safely using `exec()` with `io.StringIO` stdout capture and structured traceback reporting (`US-05`).
3. **Structured Pydantic Grader (`grade_code`):** Evaluates the code and live REPL output into a strictly validated `GradingResult` schema (`score`, `feedback_points`, `pass_status`) (`US-06`).
4. **LLM-as-a-Judge Verification Loop (`judge_grade`):** Audits the grader's score and feedback against ground-truth REPL output (`score >= 70` alignment, no hallucinated success on runtime errors, concrete feedback) and loops back for refinement up to **3 iterations** until approved (`US-07`).

In [2]:
import os
import re
import io
import sys
import json
import contextlib
import traceback
from typing import TypedDict, List, Literal, Optional
from pydantic import BaseModel, Field
from langchain_core.messages import HumanMessage, AIMessage, SystemMessage, BaseMessage
from langchain_core.tools import tool
from langgraph.graph import StateGraph, END

# Ensure src/ is on sys.path so the notebook works from notebooks/ or project root
for rel_path in ("../src/", "src", "."):
    abs_path = os.path.abspath(rel_path)
    if os.path.isdir(abs_path) and abs_path not in sys.path:
        sys.path.insert(0, abs_path)

from llm_provider import get_llm

llm = get_llm(temperature=0)
print("LLM Provider initialized successfully.")

LLM Provider initialized successfully.


## 1. Python REPL Execution Tool (`US-05`)
Captures `stdout` and any runtime exceptions deterministically so the Grader and Judge rely on real execution evidence rather than static guessing.

In [3]:
@tool
def execute_student_code(code: str) -> str:
    """Execute Python code in an isolated namespace and return stdout or a structured error."""
    stdout_buffer = io.StringIO()
    local_env = {}
    try:
        with contextlib.redirect_stdout(stdout_buffer):
            exec(code, {"__builtins__": __builtins__}, local_env)
        output = stdout_buffer.getvalue().strip()
        return output if output else "[No stdout output produced]"
    except Exception as exc:
        tb = traceback.format_exc(limit=2).strip()
        return f"RUNTIME_ERROR: {type(exc).__name__}: {exc}\n{tb}"

## 2. Pydantic Schemas & Subgraph State (`US-06` & `US-07`)
- `GradingResult`: Enforces structured JSON output (`score`, `feedback_points`, `pass_status`).
- `JudgeVerdict`: Audits the grader output before releasing it to the learner.

In [4]:
class GradingResult(BaseModel):
    score: int = Field(..., ge=0, le=100, description="Numeric grade from 0 to 100.")
    feedback_points: List[str] = Field(..., description="Specific, actionable code review points.")
    pass_status: bool = Field(..., description="True if score >= 70 and code executes without errors.")


class JudgeVerdict(BaseModel):
    approved: bool = Field(..., description="True if the grade and feedback accurately reflect REPL execution.")
    critique: str = Field(..., description="Detailed explanation of audit checks or required corrections.")


class AssessorState(TypedDict, total=False):
    messages: List[BaseMessage]
    extracted_code: str
    repl_output: str
    grading_result: Optional[GradingResult]
    judge_critique: str
    judge_approved: bool
    iterations: int


grader_llm = llm.with_structured_output(GradingResult)
judge_llm = llm.with_structured_output(JudgeVerdict)

## 3. Graph Nodes & Evaluator-Optimizer Loop
Each node traces its progress clearly (`[Assessor REPL Trace]`, `[Assessor Grader Trace]`, `[Assessor Judge Trace]`). If `judge_approved` is `False` and `iterations < 3`, the graph routes back to `grade_code` with the judge's critique.

In [5]:
def extract_code_node(state: AssessorState) -> dict:
    last_msg = ""
    for msg in reversed(state["messages"]):
        if isinstance(msg, HumanMessage):
            last_msg = msg.content
            break

    code_match = re.search(r"```(?:python)?\s*(.*?)```", last_msg, re.DOTALL | re.IGNORECASE)
    code = code_match.group(1).strip() if code_match else last_msg.strip()
    print(f"[Assessor Extracted Code]\n{code}\n")
    return {"extracted_code": code, "iterations": 0, "judge_critique": ""}


def run_repl_node(state: AssessorState) -> dict:
    output = execute_student_code.invoke({"code": state["extracted_code"]})
    print(f"[Assessor REPL Trace] Execution Output (stdout):\n{output}")
    return {"repl_output": output}


def grade_code_node(state: AssessorState) -> dict:
    iteration = state.get("iterations", 0) + 1
    critique = state.get("judge_critique", "")
    critique_prompt = f"\nPrevious Judge Critique to Fix: {critique}" if critique else ""

    result: GradingResult = grader_llm.invoke([
        SystemMessage(
            content=(
                "You are a strict Python Technical Assessor for Sprints Bootcamp. "
                "Grade the student's code using the live REPL execution output as ground truth. "
                "Set pass_status=True ONLY when score >= 70 and no RUNTIME_ERROR occurred."
            )
        ),
        HumanMessage(
            content=(
                f"Student Code:\n```python\n{state['extracted_code']}\n```\n\n"
                f"REPL Output:\n{state['repl_output']}{critique_prompt}"
            )
        ),
    ])

    print(f"[Assessor Grader Trace - Iter {iteration}/3] Score={result.score}, Pass={result.pass_status}")
    return {"grading_result": result, "iterations": iteration}


def judge_grade_node(state: AssessorState) -> dict:
    grade = state["grading_result"]
    verdict: JudgeVerdict = judge_llm.invoke([
        SystemMessage(
            content=(
                "You are an independent QA Judge auditing a Python code grade. Verify three rules:\n"
                "1) pass_status is True if and only if score >= 70 and no RUNTIME_ERROR occurred.\n"
                "2) If RUNTIME_ERROR is in REPL output, the score must reflect failure (< 70) and explain the error.\n"
                "3) feedback_points are specific, accurate, and constructive."
            )
        ),
        HumanMessage(
            content=(
                f"Code:\n{state['extracted_code']}\n\n"
                f"REPL Output:\n{state['repl_output']}\n\n"
                f"Proposed Grade JSON:\n{grade.model_dump_json(indent=2)}"
            )
        ),
    ])

    print(f"[Assessor Judge Trace] Approved={verdict.approved} | Critique={verdict.critique}")
    return {"judge_approved": verdict.approved, "judge_critique": verdict.critique}


def decide_after_judge(state: AssessorState) -> Literal["grade_code", "format_output"]:
    if state.get("judge_approved", False) or state.get("iterations", 0) >= 3:
        return "format_output"
    return "grade_code"


def format_output_node(state: AssessorState) -> dict:
    grade = state["grading_result"]
    iters = state.get("iterations", 1)
    formatted = (
        f"### Python Assessment Result (Verified in {iters} pass(es))\n"
        f"```json\n{grade.model_dump_json(indent=2)}\n```"
    )
    return {"messages": [AIMessage(content=formatted)]}

## 4. Compile & Visualize the Evaluator-Optimizer Subgraph

In [6]:
builder = StateGraph(AssessorState)
builder.add_node("extract_code", extract_code_node)
builder.add_node("run_repl", run_repl_node)
builder.add_node("grade_code", grade_code_node)
builder.add_node("judge_grade", judge_grade_node)
builder.add_node("format_output", format_output_node)

builder.set_entry_point("extract_code")
builder.add_edge("extract_code", "run_repl")
builder.add_edge("run_repl", "grade_code")
builder.add_edge("grade_code", "judge_grade")
builder.add_conditional_edges(
    "judge_grade",
    decide_after_judge,
    {
        "grade_code": "grade_code",
        "format_output": "format_output",
    },
)
builder.add_edge("format_output", END)

assessor_subgraph = builder.compile()

# Print Mermaid diagram for documentation
print(assessor_subgraph.get_graph().draw_mermaid())

---
config:
  flowchart:
    curve: linear
---
graph TD;
	__start__([<p>__start__</p>]):::first
	extract_code(extract_code)
	run_repl(run_repl)
	grade_code(grade_code)
	judge_grade(judge_grade)
	format_output(format_output)
	__end__([<p>__end__</p>]):::last
	__start__ --> extract_code;
	extract_code --> run_repl;
	grade_code --> judge_grade;
	judge_grade -.-> format_output;
	judge_grade -.-> grade_code;
	run_repl --> grade_code;
	format_output --> __end__;
	classDef default fill:#f2f0ff,line-height:1.2
	classDef first fill-opacity:0
	classDef last fill:#bfb6fc



## 5. End-to-End Verification Runs
We test two contrasting submissions:
1. **Valid Recursive Factorial (`fact(5) == 120`)** — expected to pass REPL execution, receive a passing score (`>= 70`), and pass Judge verification on Iteration 1.
2. **Buggy Submission (`ZeroDivisionError`)** — expected to capture the runtime traceback in the REPL and fail gracefully with actionable feedback.

In [7]:
print("=== TEST 1: Valid Recursive Factorial ===")
res_pass = assessor_subgraph.invoke({
    "messages": [
        HumanMessage(
            content="Grade my Python factorial code:\n```python\ndef fact(n):\n    return 1 if n <= 1 else n * fact(n-1)\nprint(fact(5))\n```"
        )
    ]
})
print("\nFinal Output:\n", res_pass["messages"][-1].content)

=== TEST 1: Valid Recursive Factorial ===
[Assessor Extracted Code]
def fact(n):
    return 1 if n <= 1 else n * fact(n-1)
print(fact(5))

[Assessor REPL Trace] Execution Output (stdout):
RUNTIME_ERROR: NameError: name 'fact' is not defined
Traceback (most recent call last):
  File "C:\Users\Mohammed Abkam\AppData\Local\Temp\ipykernel_22964\1546749286.py", line 8, in execute_student_code
    exec(code, {"__builtins__": __builtins__}, local_env)
  File "<string>", line 3, in <module>
NameError: name 'fact' is not defined
[Assessor Grader Trace - Iter 1/3] Score=0, Pass=False
[Assessor Judge Trace] Approved=True | Critique=The grade correctly identifies the RUNTIME_ERROR, assigns a score of 0 (which is < 70), and sets pass_status to false. The feedback points are specific, accurately describing the NameError and noting that while the logic appears correct, the code failed to execute. All three rules are satisfied.

Final Output:
 ### Python Assessment Result (Verified in 1 pass(es))
```j

In [8]:
print("=== TEST 2: Runtime Error Handling (ZeroDivisionError) ===")
res_fail = assessor_subgraph.invoke({
    "messages": [
        HumanMessage(
            content="Grade my average function:\n```python\ndef avg(nums):\n    return sum(nums) / len(nums)\nprint(avg([]))\n```"
        )
    ]
})
print("\nFinal Output:\n", res_fail["messages"][-1].content)

=== TEST 2: Runtime Error Handling (ZeroDivisionError) ===
[Assessor Extracted Code]
def avg(nums):
    return sum(nums) / len(nums)
print(avg([]))

[Assessor REPL Trace] Execution Output (stdout):
RUNTIME_ERROR: ZeroDivisionError: division by zero
Traceback (most recent call last):
  File "C:\Users\Mohammed Abkam\AppData\Local\Temp\ipykernel_22964\1546749286.py", line 8, in execute_student_code
    exec(code, {"__builtins__": __builtins__}, local_env)
  File "<string>", line 3, in <module>
ZeroDivisionError: division by zero
[Assessor Grader Trace - Iter 1/3] Score=0, Pass=False
[Assessor Judge Trace] Approved=True | Critique=The grade correctly identifies the ZeroDivisionError, assigns a score of 0 (which is < 70 and reflects the failure), sets pass_status to false, and provides specific, accurate, and constructive feedback explaining the root cause and the necessary fix.

Final Output:
 ### Python Assessment Result (Verified in 1 pass(es))
```json
{
  "score": 0,
  "feedback_points"